# Хакатон: прогноз продаж - старт

Отсюда начинается работа: бейзлайны, первая модель и локальная сборка сабмита (финальный сабмит собирает `06_final.ipynb`). Структура репозитория - в `README.md`, описание модулей - в `docs/`.

Прогноз - **недельный**: 9 недель с понедельника 2026-07-27, строка = неделя × склад × товар. Учиться можно только на данных по 2026-07-26 включительно - `load_data()` сам отрезает всё, что позже.

Как устроено:
- `make_forecast(sales, stock, method, **params)` - точка входа для бэктеста и локального сабмита, метод выбирается по имени. Финальный сабмит и контейнер считают прогноз через `Forecaster` (`src/forecaster.py`) с теми же признаками и тем же решением 0 или k.
- `backtest(method, **params)` - "отматываем время назад" к трём датам отсечки (`FOLDS`), прогнозируем 9 недель, сравниваем с фактом; метрики - ровно как в `score.py` из ТЗ. Результаты копятся в `outputs/experiments_log.csv`, `journal()` показывает по строке на эксперимент.
- Модель - две ступени: классификатор "будет ли продажа" (`03_classifier.ipynb`) и регрессор "сколько" (`04_regressor.ipynb`), у каждого свой набор признаков. Здесь они собираются вместе - метод `pipeline`.

Запускать можно здесь или на Kaggle GPU: `make run NB=01` из папки проекта - выводы вернутся в этот же ноутбук, а сабмит, собранный на Kaggle, ляжет в `outputs/kaggle/01_start/`.

## Метрика

Половина скора - SMAPE по каждой паре: неделя "факт 0, прогноз 0" - ошибка 0, "факт 0, прогноз хоть 0.01" - 200 %. Поэтому там, где продаж скорее всего не будет, ставим ровно 0, а не "пыль". `Bias` (10 %) - противовес: если обнулять всё подряд, по таким парам он 100 %. WAPE и RMSE (35 %) решают ходовые товары.

Итог на хакатоне - ранговый (место среди участников по каждой метрике). Локально мест не посчитать, поэтому `score` в журнале - min-max относительно наивного прогноза: 0 = идеал, 1 = не лучше, чем "как на прошлой неделе". Подробнее - в `README.md`.

## 0. Настройки и данные

`load_data()` отдаёт данные только по `CUTOFF` = 2026-07-26: дальше по правилам использовать нельзя.

In [ ]:
%pip install pandas numpy matplotlib seaborn catboost holidays

In [ ]:
import sys; sys.path.insert(0, "../src")   # код проекта - в src/

import numpy as np
import pandas as pd
from core import *

# правки в src/ подхватываются без перезапуска ядра (методы, объявленные в ноутбуке, после этого перезапусти);
# на Kaggle не включаем: там старый IPython, и это расширение на новом Python не грузится
if not ON_KAGGLE:
    %load_ext autoreload
    %autoreload 2

sales, stock = load_data()
print("sales:", sales.shape, sales.date.min().date(), "...", sales.date.max().date())
print("stock:", stock.shape, stock.date.min().date(), "...", stock.date.max().date())
sales.head()

## 1. Бейзлайны

In [ ]:
pd.concat([
    backtest("naive_last"),
    backtest("zeros"),
    backtest("mean", window=8),
    backtest("median", window=8),
])

## 2. Свои эксперименты

Меняем одну вещь за раз. Главный фолд - `season2025` (тот же сезон, что и тест, в журнале `score_season2025`). Изменение берём, если оно не хуже на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005. Новый метод - функция с `@register("имя")`, после этого `backtest("имя")` уже работает.

In [ ]:
# порог "мелочь -> 0" и общий множитель - параметры make_forecast, работают для любого метода
backtest("mean", window=8, zero_below=1, note="мелочь -> 0")
backtest("mean", window=8, zero_below=1, scale=1.15, note="мелочь -> 0, множитель")
journal().head(10)

In [ ]:
# шаблон нового метода
@register("my_method")
def my_method(sales, stock, pairs, weeks, window=8):
    hist = weekly_history(sales, pairs, window)   # пары × последние window недель
    values = hist.mean(axis=1)                    # TODO: своя логика
    return constant(values, weeks)

backtest("my_method", window=8, note="шаблон")

## 3. Итоговая модель: классификатор + регрессор

Классификатор (`03_classifier.ipynb`) решает, будет ли продажа на неделе, регрессор (`04_regressor.ipynb`) - сколько; где вероятность ниже порога - ровно 0. Здесь проверяем их вместе и подбираем порог. Для сравнения - каждая модель по отдельности (`classifier`: классификатор + "обычное количество"; `regressor`: простое правило + регрессор) и первый сабмит.

Первый запуск обучает классификатор и регрессор на каждом фолде - минут 5 локально; дальше модели берутся из кэша.

In [ ]:
import pipeline   # метод "pipeline"; заодно подключает "classifier" и "regressor"

for t in [0.5, 0.55, 0.6, 0.65]:
    backtest("pipeline", threshold=t, note="итоговая модель")
backtest("classifier", threshold=0.55, note="только классификатор")
backtest("regressor", note="только регрессор")
j = journal()
j[j.note.isin(["итоговая модель", "только классификатор", "только регрессор"]) | (j.params == "{'window': 8, 'zero_below': 1, 'scale': 1.15}")]   # + первый сабмит

## 4. Откуда ошибка итоговой модели

Разбор на главном фолде - подсказывает, какую ступень улучшать:
- `kinds` - из чего складываются SMAPE и WAPE: "лишняя продажа" (сказали "будет", а не было - ошибка классификатора), "пропущенная продажа" (сказали 0, а была - тоже классификатор), "ошибка количества" (обе ненулевые - регрессор). Вклады в сумме дают итоговую метрику.
- `by_category`, `by_warehouse` - какие группы пар дают больше всего ошибки и в какую сторону перекос (прогноз / факт).
- `worst_pairs` - пары с наибольшей ошибкой в штуках: они решают WAPE и RMSE.

In [ ]:
err = error_breakdown("pipeline", fold="season2025", threshold=0.55)
display(err["kinds"])
display(err["by_category"])
display(err["by_warehouse"])
err["worst_pairs"]

## 5. Сабмит локально

Итоговый конфиг из `src/final.py` (два CatBoost и LightGBM, порог 0.55 для недель 1-4 и 0.5 для недель 5-9, выученное соответствие прогноз -> целое) через `make_forecast`. Финальный сабмит собирает `06_final.ipynb`, здесь проверяются формат и детерминированность. Простой запасной вариант оставлен закомментированной строкой ниже. Модели обучаются на всей истории по 2026-07-26 (несколько минут), проверка детерминированности обучает их второй раз с нуля. Файл - `outputs/submission.csv`.

Набор пар - все 11 × 799. Если в эталоне окажутся только пары с продажами, `sub` нужно отфильтровать перед сохранением и передать эти пары в `check_submission`.

In [ ]:
import pipeline, classifier, regressor

import final
METHOD, PARAMS = "pipeline", final.PARAMS        # <- итоговый конфиг (src/final.py); финальный сабмит - notebooks/06_final
# METHOD, PARAMS = "mean", {"window": 8, "zero_below": 1, "scale": 1.15}   # простой запасной вариант

sub = make_forecast(sales, stock, METHOD, **PARAMS)
print(check_submission(sub) or "формат ок")
classifier._CACHE.clear(); regressor._CACHE.clear()   # иначе второй прогноз возьмёт модели из кэша и проверка ничего не проверит
print("детерминированность:", sub.equals(make_forecast(sales, stock, METHOD, **PARAMS)))
sub.to_csv(SUBMISSION_CSV, index=False, date_format="%Y-%m-%d")
print(len(sub), "строк, недели", sub.week.min().date(), "...", sub.week.max().date(), "->", SUBMISSION_CSV)